# 33 - LOSO (Leave-One-Subject-Out) Cross-Validation

**Tujuan:** Evaluasi robustness model menggunakan LOSO — setiap user menjadi test set 1x.

**37 fold** × 3 model terbaik (front-only 4-class):
1. Intermediate Fusion TL B1 (best: 0.412)
2. Late Fusion B3 (best: 0.394)
3. FCNN B3 (best: 0.361)

**Output:** Mean ± Std Macro F1 per model

In [ ]:
import subprocess
import sys

# Run LOSO for top 3 models (4-class)
print("Running LOSO for Intermediate Fusion TL...")
subprocess.run([sys.executable, "../scripts/run_loso.py",
    "--models", "intermediate_tl",
    "--num-classes", "4"], check=True)

print("\n\nRunning LOSO for Late Fusion...")
subprocess.run([sys.executable, "../scripts/run_loso.py",
    "--models", "late_fusion",
    "--num-classes", "4"], check=True)

print("\n\nRunning LOSO for FCNN...")
subprocess.run([sys.executable, "../scripts/run_loso.py",
    "--models", "fcnn",
    "--num-classes", "4"], check=True)

## Ringkasan LOSO

In [ ]:
import json
import numpy as np
from pathlib import Path

loso_dir = Path("../models/frontonly/loso")

print("=" * 75)
print("RINGKASAN LOSO CROSS-VALIDATION (4-CLASS FRONT-ONLY)")
print("=" * 75)
print(f"{'Model':<35} {'Macro F1':>15} {'Accuracy':>15} {'Folds':>8}")
print("-" * 75)

for f in sorted(loso_dir.glob("loso_*_4class.json")):
    data = json.load(open(f))
    name = data['description']
    mf1 = data['macro_f1_mean']
    sf1 = data['macro_f1_std']
    ma = data['accuracy_mean']
    sa = data['accuracy_std']
    n = data['num_folds']
    print(f"{name:<35} {mf1:.4f} +/- {sf1:.4f}  {ma:.4f} +/- {sa:.4f}  {n:>5}")

print()
print("Perbandingan dengan Single Split:")
print("  Intermediate TL B1 single split: 0.412")
print("  Late Fusion B3 single split:     0.394")
print("  FCNN B3 single split:            0.361")

In [ ]:
# Per-fold detail
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, f in zip(axes, sorted(loso_dir.glob("loso_*_4class.json"))):
    data = json.load(open(f))
    folds = data['per_fold']
    users = [r['test_user'] for r in folds]
    f1s = [r['macro_f1'] for r in folds]
    
    ax.barh(range(len(users)), f1s, color='steelblue', alpha=0.8)
    ax.set_yticks(range(len(users)))
    ax.set_yticklabels(users, fontsize=7)
    ax.axvline(x=np.mean(f1s), color='red', linestyle='--', label=f'Mean: {np.mean(f1s):.3f}')
    ax.set_xlabel('Macro F1')
    ax.set_title(data['description'][:30])
    ax.legend(fontsize=8)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(str(loso_dir / 'loso_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Chart saved to {loso_dir / 'loso_comparison.png'}")